In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
import ccxt
from itertools import combinations
from statsmodels.tsa.stattools import coint


In [2]:
UNIVERSE = [
    "BTC",
    "ETH",
    "SOL",
    "BNB",
    "XRP",
    "DOGE",
    "ADA",
    "LINK",
    "AVAX",
    "DOT",
    "LTC",
    "ATOM",
    "AAVE",
    "UNI",
    "ARB",
    "OP",
    "SUI",
    "NEAR",
    "ALGO",
    "FIL"
]

In [3]:
SYMBOLS = [f"{coin}/USD" for coin in UNIVERSE]

In [4]:
exchange = ccxt.kraken()
markets = exchange.load_markets()

symbols = []

for coin in UNIVERSE:
    pair = f"{coin}/USD"
    if pair in markets:
        symbols.append(pair)

print(symbols)

['BTC/USD', 'ETH/USD', 'SOL/USD', 'BNB/USD', 'XRP/USD', 'DOGE/USD', 'ADA/USD', 'LINK/USD', 'AVAX/USD', 'DOT/USD', 'LTC/USD', 'ATOM/USD', 'AAVE/USD', 'UNI/USD', 'ARB/USD', 'OP/USD', 'SUI/USD', 'NEAR/USD', 'ALGO/USD', 'FIL/USD']


In [5]:
pairs = list(combinations(symbols, 2))

In [11]:
def ohlcv_to_df(data, symbol):
    df = pd.DataFrame(data, columns=["timestamp", "open", "high", "low", "close", "volume"])
    df["timestamp"] = pd.to_datetime(df["timestamp"], unit="ms")
    df = df[["timestamp", "close"]].copy()
    df = df.rename(columns={"close": symbol})
    return df

In [43]:
def test_pairs(pair, window):
    A_raw = pair[0]
    B_raw = pair[1]

    A = exchange.fetch_ohlcv(A_raw, timeframe="1h", limit=1000)
    B = exchange.fetch_ohlcv(B_raw, timeframe="1h", limit=1000)

    A_name = A_raw.split("/")[0]
    B_name = B_raw.split("/")[0]

    A = ohlcv_to_df(A, A_name)
    B = ohlcv_to_df(B, B_name)
    
    log_price_A = np.log(A[A_name]).diff().dropna()
    log_price_B = np.log(B[B_name]).diff().dropna()

    cor = log_price_A.corr(log_price_B)

    score, pvalue, _ = coint(log_price_A, log_price_B)

    X = sm.add_constant(log_price_A)
    y = log_price_B

    ols = sm.OLS(y, X).fit()
    # print(ols.summary())

    beta = ols.params[ols.params.index[1]]
    spread = ols.resid
    spread_vol = spread.std()

    spread_lag = spread.shift(1)
    spread_diff = spread - spread_lag
    df = pd.concat([spread_diff, spread_lag], axis=1).dropna()
    df.columns = ["diff", "lag"]
    model = sm.OLS(df["diff"], sm.add_constant(df["lag"])).fit()
    lam = model.params["lag"]
    half_life = -np.log(2) / lam

    spread_mean = spread.rolling(window).mean()
    spread_std = spread.rolling(window).std()
    z = (spread - spread_mean) / spread_std

    upper = ((z.shift(1) < 2) &(z >= 2)).sum()
    lower = ((z.shift(1) > -2) & (z <= -2)).sum()
    crossings = upper + lower

    df = {
        "Pair": f"{A_name}/{B_name}",
        "Corr": cor,
        "Score": score,
        "P-value": pvalue,
        "Beta": beta,
        "Spread Vol": spread_vol,
        "Half-life": half_life,
        "Z-score": z,
        "Crossings": crossings
    }
    
    return df
    

In [44]:
df_list = []

In [ ]:
for pair in pairs:
    df = test_pairs(pair, 25)
    df_list.append(df)

In [ ]:
final_df = pd.concat(df_list, ignore_index=True)

In [ ]:
result = final_df.query('Corr > 0.8')
result = result.query('P-value < 0.05')

In [ ]:
result = result.sort_values(by="Half-life")
result = result.sort_values(by="Crossings", ascending=False)

In [ ]:
# Now rank the pairs: compute statistics like
        # half-life
        # spread volatility
        # average z-score crossings